# IndicTrans2 LoRA — Clean Evaluation Notebook (checkpoint-5500)

**Single purpose:** load the saved LoRA checkpoint from Drive and evaluate it against the base
model. No training code, no redefinitions, no leftover state from other sessions.

**Run this in a freshly started/restarted runtime** — `Runtime -> Restart session` first if you've
run anything else in this session, so there's no leftover GPU memory or stale variables from a
previous training run.


## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Pin transformers to a version IndicTransToolkit's collator.py is compatible with.
# (Newer transformers moved PreTrainedTokenizerBase out of transformers.tokenization_utils,
#  which breaks IndicTransToolkit's import.)
!pip install -q "transformers==4.45.2"
!pip install -q IndicTransToolkit
!pip install -q accelerate peft sacrebleu sentencepiece datasets
!pip install -q indic-nlp-library sacremoses


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.4/548.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.0/128.0 kB 14.6 MB/s eta 0:0

> If Colab prompts you to restart the session after the installs above, do so now,
> then continue from the next cell. Do **not** re-run the pip install cell after restarting —
> just continue below.

In [ ]:
# Hugging Face auth — ONLY needed if the model repo requires it (it usually doesn't for
# ai4bharat/indictrans2-en-indic-dist-200M, which is public). If you previously hit a
# GatedRepoError, that is almost always caused by a stray/invalid token in the environment
# rather than the repo actually being private — clear it first:
import os
os.environ.pop("HF_TOKEN", None)
os.environ.pop("HUGGING_FACE_HUB_TOKEN", None)

from huggingface_hub import logout
try:
    logout()
except Exception:
    pass

# If you DO need to authenticate (e.g. for a private/gated model), use Colab Secrets —
# never hardcode a token in a cell. Set it once via the key icon in the left sidebar.
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get('HF_TOKEN'))


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import login

hf_token = "your hf token"
login(token=hf_token)


In [ ]:
import time
import random
import torch
import sacrebleu
from tqdm.auto import tqdm
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
from IndicTransToolkit.processor import IndicProcessor

SEED = 42
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32
print("Device:", device, "| dtype:", dtype)


Device: cuda | dtype: torch.float16


In [ ]:
# ---- Config ----
import os
PROJECT_DIR = "/content/drive/MyDrive/NMT_Project_Conv"
OUTPUT_DIR = os.path.join(PROJECT_DIR, "indictrans2_lora_conv")
CHECKPOINT = os.path.join(OUTPUT_DIR, "checkpoint-6250")
VAL_RAW_PATH = os.path.join(PROJECT_DIR, "val_raw_conv")

BASE_MODEL = "ai4bharat/indictrans2-en-indic-dist-200M"
SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"
MAX_TOKEN_LENGTH = 128

print("Checkpoint exists:", os.path.exists(CHECKPOINT))
print("Validation set exists:", os.path.exists(VAL_RAW_PATH))


Checkpoint exists: True
Validation set exists: True


In [ ]:
!pip install "torchao>0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 12.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## 2. Load Models

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
ip = IndicProcessor(inference=True)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL, trust_remote_code=True, torch_dtype=dtype
).to(device)
base_model.eval()

# Separate base-model copy for the LoRA adapter (PeftModel wraps its own base internally,
# but loading a fresh copy avoids any chance of the two evaluations sharing mutated state)
lora_base = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL, trust_remote_code=True, torch_dtype=dtype
).to(device)

lora_model = PeftModel.from_pretrained(lora_base, CHECKPOINT).to(device)
lora_model.eval()

print("Base model device:  ", next(base_model.parameters()).device)
print("LoRA model device:  ", next(lora_model.parameters()).device)
print("Base use_cache:     ", base_model.config.use_cache)
print("LoRA use_cache:     ", lora_model.config.use_cache)

# Training can leave use_cache=False on a model config (needed for gradient checkpointing).
# Force it back on for generation — this alone can be a 10-50x speed difference.
base_model.config.use_cache = True
lora_model.config.use_cache = True


Base model device:   cuda:0
LoRA model device:   cuda:0
Base use_cache:      True
LoRA use_cache:      True


In [ ]:
!nvidia-smi


## 3. Translation Helper (single definition, used everywhere below)

In [ ]:
def batch_translate(model, tokenizer, ip, sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
                     batch_size=16, num_beams=1, max_length=MAX_TOKEN_LENGTH, desc="Translating"):
    model.eval()
    use_amp = model.device.type == "cuda"
    outputs = []
    total_batches = (len(sentences) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(sentences), batch_size), total=total_batches, desc=desc, unit="batch"):
        batch = sentences[i:i + batch_size]
        preproc = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
        inputs = tokenizer(preproc, truncation=True, padding=True,
                            max_length=max_length, return_tensors="pt").to(model.device)
        with torch.no_grad():
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    generated = model.generate(**inputs, num_beams=num_beams,
                                                max_length=max_length, use_cache=True)
            else:
                generated = model.generate(**inputs, num_beams=num_beams,
                                            max_length=max_length, use_cache=True)
        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(ip.postprocess_batch(decoded, lang=tgt_lang))
    return outputs


In [ ]:
def batch_translate(model, tokenizer, ip, sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG,
                    batch_size=16, num_beams=5, max_length=MAX_TOKEN_LENGTH, desc="Translating"):
    model.eval()
    outputs = []
    total_batches = (len(sentences) + batch_size - 1) // batch_size

    # Enforce clear generation configs directly inside generation loop
    gen_config = {
        "num_beams": num_beams,
        "max_length": max_length,
        "use_cache": True,
        "no_repeat_ngram_size": 3,  # Prevents repetitive looping phrases
        "early_stopping": True
    }

    for i in tqdm(range(0, len(sentences), batch_size), total=total_batches, desc=desc, unit="batch"):
        batch = sentences[i:i + batch_size]
        preproc = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)

        inputs = tokenizer(
            preproc,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            if model.device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    generated = model.generate(**inputs, **gen_config)
            else:
                generated = model.generate(**inputs, **gen_config)

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(ip.postprocess_batch(decoded, lang=tgt_lang))
    return outputs


## 4. Sanity Check — Single Sentence

Run this before anything else. If this hangs or takes more than a few seconds on GPU,
stop here and debug (check `nvidia-smi`, check `use_cache`, check device placement above) —
don't proceed to the full validation run until this is fast.

In [ ]:
test_sentence = ["What's up, dude? Long time no see."]

t0 = time.time()
out = batch_translate(lora_model, tokenizer, ip, test_sentence, batch_size=1, num_beams=1,
                       desc="Sanity check")
print(f"Took {time.time() - t0:.2f}s")
print("LoRA output:", out[0])


Sanity check:   0%|          | 0/1 [00:00<?, ?batch/s]

Took 0.51s
LoRA output: क्या बात है, यार? लंबे समय तक कोई नहीं देखा.


## 5. Load Validation Set

In [ ]:
val_raw = load_from_disk(VAL_RAW_PATH)
val_en = [ex["en"] for ex in val_raw["translation"]]
val_hi_ref = [ex["hi"] for ex in val_raw["translation"]]

print("Validation examples:", len(val_en))
print("Sample EN:", val_en[0])
print("Sample HI:", val_hi_ref[0])


Validation examples: 1000
Sample EN: If the transparency key has the value COLOR, then this key determines the color which is used for indicating transparency.
Sample HI: यदि पारदर्शी कुंजी में रंग मूल्य है, तब यह कुंजी निर्धारित करती है कि कौन सा रंग पारदर्शिता को प्रदर्शित करने में उपयोग में लिया जाएगा.


## 6. Quick 200-Sentence Evaluation

Run this first as a fast sanity check before committing to the full validation set.

In [ ]:
N_EVAL = 200
NUM_BEAMS = 5  # was 1 — switch to beam search to match your original IITB baseline's decoding setting
EVAL_BATCH_SIZE = 32 if device.type == "cuda" else 4
eval_en = val_en[:N_EVAL]
eval_ref = val_hi_ref[:N_EVAL]

t0 = time.time()
base_predictions_200 = batch_translate(base_model, tokenizer, ip, eval_en,
                                        batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                        desc="Base (200)")
print(f"Base translation took {(time.time()-t0)/60:.2f} min")

t0 = time.time()
lora_predictions_200 = batch_translate(lora_model, tokenizer, ip, eval_en,
                                        batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                        desc="LoRA (200)")
print(f"LoRA translation took {(time.time()-t0)/60:.2f} min")

Base (200):   0%|          | 0/7 [00:00<?, ?batch/s]

Base translation took 0.31 min


LoRA (200):   0%|          | 0/7 [00:00<?, ?batch/s]

LoRA translation took 0.40 min


In [ ]:
#### beamsincreased
base_bleu_200 = sacrebleu.corpus_bleu(base_predictions_200, [eval_ref])
base_chrf_200 = sacrebleu.corpus_chrf(base_predictions_200, [eval_ref], word_order=2)
lora_bleu_200 = sacrebleu.corpus_bleu(lora_predictions_200, [eval_ref])
lora_chrf_200 = sacrebleu.corpus_chrf(lora_predictions_200, [eval_ref], word_order=2)

print("=" * 60)
print(f"Base BLEU:   {base_bleu_200.score:.2f}  ->  LoRA BLEU:   {lora_bleu_200.score:.2f}  ({lora_bleu_200.score - base_bleu_200.score:+.2f})")
print(f"Base chrF++: {base_chrf_200.score:.2f}  ->  LoRA chrF++: {lora_chrf_200.score:.2f}  ({lora_chrf_200.score - base_chrf_200.score:+.2f})")
print("=" * 60)


Base BLEU:   13.06  ->  LoRA BLEU:   17.58  (+4.52)
Base chrF++: 34.42  ->  LoRA chrF++: 38.35  (+3.93)


## 7. Full Validation Set Evaluation

Run only after Section 6 looks correct and ran at a reasonable speed.

In [ ]:
N_EVAL = len(val_en)
eval_en = val_en
eval_ref = val_hi_ref

t0 = time.time()
base_predictions = batch_translate(base_model, tokenizer, ip, eval_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                    desc="Base (full)")
print(f"Base translation took {(time.time()-t0)/60:.2f} min")

t0 = time.time()
lora_predictions = batch_translate(lora_model, tokenizer, ip, eval_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                    desc="LoRA (full)")
print(f"LoRA translation took {(time.time()-t0)/60:.2f} min")


Base (full):   0%|          | 0/32 [00:00<?, ?batch/s]

Base translation took 1.65 min


LoRA (full):   0%|          | 0/32 [00:00<?, ?batch/s]

LoRA translation took 1.85 min


In [ ]:
#### beams inc
N_EVAL = len(val_en)
eval_en = val_en
eval_ref = val_hi_ref

t0 = time.time()
base_predictions = batch_translate(base_model, tokenizer, ip, eval_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                    desc="Base (full)")
print(f"Base translation took {(time.time()-t0)/60:.2f} min")

t0 = time.time()
lora_predictions = batch_translate(lora_model, tokenizer, ip, eval_en,
                                    batch_size=EVAL_BATCH_SIZE, num_beams=NUM_BEAMS,
                                    desc="LoRA (full)")
print(f"LoRA translation took {(time.time()-t0)/60:.2f} min")


Base (full):   0%|          | 0/32 [00:00<?, ?batch/s]

Base translation took 1.66 min


LoRA (full):   0%|          | 0/32 [00:00<?, ?batch/s]

LoRA translation took 1.94 min


In [ ]:
base_bleu = sacrebleu.corpus_bleu(base_predictions, [eval_ref])
base_chrf = sacrebleu.corpus_chrf(base_predictions, [eval_ref], word_order=2)
lora_bleu = sacrebleu.corpus_bleu(lora_predictions, [eval_ref])
lora_chrf = sacrebleu.corpus_chrf(lora_predictions, [eval_ref], word_order=2)

print("=" * 60)
print(f"FULL VALIDATION SET ({N_EVAL} sentences)")
print(f"Base BLEU:   {base_bleu.score:.2f}  ->  LoRA BLEU:   {lora_bleu.score:.2f}  ({lora_bleu.score - base_bleu.score:+.2f})")
print(f"Base chrF++: {base_chrf.score:.2f}  ->  LoRA chrF++: {lora_chrf.score:.2f}  ({lora_chrf.score - base_chrf.score:+.2f})")
print("=" * 60)

import json
results = {
    "checkpoint": CHECKPOINT,
    "n_eval": N_EVAL,
    "num_beams": NUM_BEAMS,
    "base_bleu": base_bleu.score,
    "base_chrf": base_chrf.score,
    "lora_bleu": lora_bleu.score,
    "lora_chrf": lora_chrf.score,
}
with open(os.path.join(PROJECT_DIR, "eval_results_checkpoint6000.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))


FULL VALIDATION SET (1000 sentences)
Base BLEU:   13.30  ->  LoRA BLEU:   16.61  (+3.31)
Base chrF++: 33.49  ->  LoRA chrF++: 36.97  (+3.48)
{
  "checkpoint": "/content/drive/MyDrive/NMT_Project_Conv/indictrans2_lora_conv/checkpoint-6250",
  "n_eval": 1000,
  "num_beams": 5,
  "base_bleu": 13.299960315075303,
  "base_chrf": 33.494087926881974,
  "lora_bleu": 16.60880087296448,
  "lora_chrf": 36.97433009327248
}


In [ ]:

###### beams inc
base_bleu = sacrebleu.corpus_bleu(base_predictions, [eval_ref])
base_chrf = sacrebleu.corpus_chrf(base_predictions, [eval_ref], word_order=2)
lora_bleu = sacrebleu.corpus_bleu(lora_predictions, [eval_ref])
lora_chrf = sacrebleu.corpus_chrf(lora_predictions, [eval_ref], word_order=2)

print("=" * 60)
print(f"FULL VALIDATION SET ({N_EVAL} sentences)")
print(f"Base BLEU:   {base_bleu.score:.2f}  ->  LoRA BLEU:   {lora_bleu.score:.2f}  ({lora_bleu.score - base_bleu.score:+.2f})")
print(f"Base chrF++: {base_chrf.score:.2f}  ->  LoRA chrF++: {lora_chrf.score:.2f}  ({lora_chrf.score - base_chrf.score:+.2f})")
print("=" * 60)
import json
results = {
    "checkpoint": CHECKPOINT,
    "n_eval": N_EVAL,
    "num_beams": NUM_BEAMS,
    "base_bleu": base_bleu.score,
    "base_chrf": base_chrf.score,
    "lora_bleu": lora_bleu.score,
    "lora_chrf": lora_chrf.score,
}
with open(os.path.join(PROJECT_DIR, "eval_results_checkpoint6000.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

FULL VALIDATION SET (1000 sentences)
Base BLEU:   13.30  ->  LoRA BLEU:   16.61  (+3.31)
Base chrF++: 33.49  ->  LoRA chrF++: 36.97  (+3.48)
{
  "checkpoint": "/content/drive/MyDrive/NMT_Project_Conv/indictrans2_lora_conv/checkpoint-6250",
  "n_eval": 1000,
  "num_beams": 5,
  "base_bleu": 13.299960315075303,
  "base_chrf": 33.494087926881974,
  "lora_bleu": 16.60880087296448,
  "lora_chrf": 36.97433009327248
}


## 8. Qualitative Spot Check

In [ ]:
print("Sample translations (EN | Reference HI | Base | LoRA)\n")
for i in random.sample(range(len(eval_en)), k=min(5, len(eval_en))):
    print(f"EN:   {eval_en[i]}")
    print(f"REF:  {eval_ref[i]}")
    print(f"BASE: {base_predictions[i]}")
    print(f"LORA: {lora_predictions[i]}")
    print("-" * 100)


Sample translations (EN | Reference HI | Base | LoRA)

EN:   Number of terms:
REF:  शर्तों की संख्या
BASE: शब्दों की संख्याः
LORA: शब्दों की संख्याः
----------------------------------------------------------------------------------------------------
EN:   They're old, and their control units are unpredictable.
REF:  ये पुरानी है, और इनके Cotrol Unit का कोई भरोसा नहीं .
BASE: वे पुराने हैं, और उनकी नियंत्रण इकाइयाँ अप्रत्याशित हैं।
LORA: वे पुराने हैं, और उनके नियंत्रण इकाइयों अप्रत्याशित हैं.
----------------------------------------------------------------------------------------------------
EN:   it's wrong to take the fruit
REF:  फल लेना गलत है.
BASE: फल लेना गलत है
LORA: फल लेना गलत है
----------------------------------------------------------------------------------------------------
EN:   Reset Font
REF:  फ़ॉन्ट चुनें
BASE: फ़ॉन्ट रीसेट करें
LORA: फ़ॉन्ट रीसेट करें
----------------------------------------------------------------------------------------------------
EN:   Those who 

## 9. Adversarial / Conversational Test Sentences

Hand-picked informal sentences to probe whether the conversational fine-tune actually
shifted the model's register (slang, contractions, idioms).

In [ ]:
adversarial_sentences = [
    "What's up, dude? Long time no see.",
    "I'm gonna grab some grub, you in?",
    "My bad, I didn't mean to mess it up.",
    "Get out of here! You're kidding me.",
    "Chill out, it's not a big deal.",
]

base_adv = batch_translate(base_model, tokenizer, ip, adversarial_sentences,
                            batch_size=8, num_beams=1, desc="Base (adversarial)")
lora_adv = batch_translate(lora_model, tokenizer, ip, adversarial_sentences,
                            batch_size=8, num_beams=1, desc="LoRA (adversarial)")

print("=" * 70)
for i, en in enumerate(adversarial_sentences):
    print(f"EN:   {en}")
    print(f"BASE: {base_adv[i]}")
    print(f"LORA: {lora_adv[i]}")
    print("-" * 70)


Base (adversarial):   0%|          | 0/1 [00:00<?, ?batch/s]

LoRA (adversarial):   0%|          | 0/1 [00:00<?, ?batch/s]

EN:   What's up, dude? Long time no see.
BASE: क्या हुआ यार? बहुत दिनों से नहीं देख रहा हूँ।
LORA: क्या बात है, यार? लंबे समय तक कोई नहीं देखा.
----------------------------------------------------------------------
EN:   I'm gonna grab some grub, you in?
BASE: मैं कुछ गुस्सा लेने जा रहा हूँ, आप अंदर?
LORA: मैं कुछ ग्रब पकड़ रहा हूँ, आप अंदर?
----------------------------------------------------------------------
EN:   My bad, I didn't mean to mess it up.
BASE: मेरी बुराई है, मेरा इरादा इसे गड़बड़ करना नहीं था।
LORA: मेरी बुराई, मैं इसे गड़बड़ करने के लिए नहीं करना चाहता था.
----------------------------------------------------------------------
EN:   Get out of here! You're kidding me.
BASE: यहाँ से निकल जाओ! तुम मेरा मजाक कर रहे हो।
LORA: तुम मेरा मजाक कर रहे हो!
----------------------------------------------------------------------
EN:   Chill out, it's not a big deal.
BASE: शांत हो जाओ, यह कोई बड़ी बात नहीं है।
LORA: शांत हो जाओ, यह कोई बड़ी बात नहीं है.
------------------------------

In [ ]:
adversarial_sentences = [
    "Listen brother, don't take tension, everything will be fine.",
    "Are you coming to the market today or should I go alone?",
    "Tell him clearly that we cannot give money right now.",
    "Please call me once you reach the metro station.",
    "Don't worry yaar, it was a minor mistake."
]

# Ensure beam configuration matches your primary evaluation configuration (num_beams=5)
base_adv = batch_translate(base_model, tokenizer, ip, adversarial_sentences, batch_size=8, num_beams=5, desc="Base Probes")
lora_adv = batch_translate(lora_model, tokenizer, ip, adversarial_sentences, batch_size=8, num_beams=5, desc="LoRA Probes")

print("=" * 80)
for i, en in enumerate(adversarial_sentences):
    print(f"ENGLISH: {en}")
    print(f"BASE LM: {base_adv[i]}")
    print(f"LORA FX: {lora_adv[i]}")
    print("-" * 80)


Base Probes:   0%|          | 0/1 [00:00<?, ?batch/s]

LoRA Probes:   0%|          | 0/1 [00:00<?, ?batch/s]

ENGLISH: Listen brother, don't take tension, everything will be fine.
BASE LM: सुनो भाई, तनाव मत लो, सब कुछ ठीक हो जाएगा।
LORA FX: सुनो भाई, तनाव मत लो, सब कुछ ठीक हो जाएगा.
--------------------------------------------------------------------------------
ENGLISH: Are you coming to the market today or should I go alone?
BASE LM: क्या आप आज बाज़ार आ रहे हैं या मुझे अकेला जाना चाहिए?
LORA FX: तुम आज बाजार आ रहे हो या मैं अकेला जाना चाहिए?
--------------------------------------------------------------------------------
ENGLISH: Tell him clearly that we cannot give money right now.
BASE LM: उसे स्पष्ट रूप से बताएँ कि हम अभी पैसे नहीं दे सकते।
LORA FX: उसे स्पष्ट रूप से कहें कि हम अभी पैसे नहीं दे सकते।
--------------------------------------------------------------------------------
ENGLISH: Please call me once you reach the metro station.
BASE LM: मेट्रो स्टेशन पहुँचने के बाद कृपया मुझे फोन करें।
LORA FX: मेट्रो स्टेशन पहुँचने के बाद कृपया मुझे फ़ोन करें.
-----------------------------------